In [2]:
!pip install pandas

In [3]:
import torch

print("PyTorch version:", torch.__version__)


PyTorch version: 2.6.0+cu124


In [4]:
import numpy as np
import pandas as pd
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split
import torch.nn.functional as F
import torchvision.models as models
import time

In [5]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
print(device)

cuda


In [7]:
# Data augmentation for training
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),  # Randomly flip the image horizontally
    transforms.RandomCrop(32, padding=4),  # Randomly crop the image
    transforms.Resize(224),  # Resize to 224x224 for ResNet
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# No augmentation for validation and test
val_test_transform = transforms.Compose([
    transforms.Resize(224),  # Resize to 224x224 for ResNet
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load CIFAR-10 dataset
train_val_dataset = datasets.CIFAR100(root='./data', train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR100(root='./data', train=False, download=True, transform=val_test_transform)

# Split train_val_dataset into train and validation sets (80% train, 20% validation)
train_size = int(0.8 * len(train_val_dataset))
val_size = len(train_val_dataset) - train_size
train_dataset, val_dataset = random_split(train_val_dataset, [train_size, val_size])

# Apply val_test_transform to the validation set
val_dataset.dataset.transform = val_test_transform

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=4)

100%|██████████| 169M/169M [00:07<00:00, 23.1MB/s] 


In [8]:
# Load pretrained ResNet-50 (Teacher Model)
teacher = models.resnet50(pretrained=True)

# Modify the final fully connected layer for 10 classes (CIFAR-100)
teacher.fc = nn.Linear(teacher.fc.in_features, 100)
# Move models to device
teacher = teacher.to(device)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 178MB/s] 


In [9]:
model_path = '/kaggle/input/teacher_res50_cifar100/pytorch/default/1/Teacher_Res50_Cifar100.pth'
# Load the model weights
teacher.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))

<All keys matched successfully>

In [10]:
# Load pretrained ResNet-18 (Student Model)
student = models.resnet18(pretrained=True)
# Modify the final fully connected layer for 100 classes (CIFAR-100)
student.fc = nn.Linear(student.fc.in_features, 100)
student = student.to(device)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 165MB/s] 


In [11]:

# model_path = '/kaggle/input/studentc10/pytorch/default/1/student_before_pruning.pth'
# # Load the model weights
# student.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))

In [12]:
# Logits normalization function
def normalize(logit):
    mean = logit.mean(dim=-1, keepdim=True)
    stdv = logit.std(dim=-1, keepdim=True)
    return (logit - mean) / (1e-7 + stdv)


In [13]:
# CA-KLD Loss for Classification
def cakld_loss(student_logits, teacher_logits, beta_prob):
    # Forward KL (student || teacher)
    student_log_prob = F.log_softmax(student_logits, dim=1)
    teacher_prob = F.softmax(teacher_logits, dim=1)
    forward_kl = F.kl_div(student_log_prob, teacher_prob, reduction='batchmean')

    # Reverse KL (teacher || student)
    teacher_log_prob = F.log_softmax(teacher_logits, dim=1)
    student_prob = F.softmax(student_logits, dim=1)
    reverse_kl = F.kl_div(teacher_log_prob, student_prob, reduction='batchmean')

    # Combined KL loss
    kl_loss = beta_prob * reverse_kl + (1 - beta_prob) * forward_kl
    return kl_loss


In [14]:
def evaluate(model, test_loader, device):
    model = model.to(device)  # Ensure model is on the correct device
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total


In [15]:
def calculate_sparsity(model):
    total_zeros = 0
    total_params = 0
    for name, param in model.named_parameters():
        if 'weight' in name:
            total_zeros += torch.sum(param == 0).item()
            total_params += param.numel()
    return total_zeros / total_params

In [16]:
import torch
import time
def measure_inference_time(model, test_loader, num_runs=5):
    device = torch.device('cpu')
    model.eval()
    model.to(device)

    # Warm-up (one batch to avoid startup cost)
    with torch.no_grad():
        for inputs, _ in test_loader:
            inputs = inputs.to(device)
            _ = model(inputs)
            break

    total_time = 0
    total_images = 0

    with torch.no_grad():
        for _ in range(num_runs):
            for inputs, _ in test_loader:
                inputs = inputs.to(device)
                batch_size = inputs.size(0)
                start_time = time.time()
                _ = model(inputs)
                end_time = time.time()

                total_time += (end_time - start_time)
                total_images += batch_size

    avg_time_per_image = total_time / total_images
    return avg_time_per_image


In [17]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def calculate_model_size(model, filename="temp.pth"):
    torch.save(model.state_dict(), filename)
    size = os.path.getsize(filename) / (1024 * 1024)  # Size in MB
    os.remove(filename)
    return size

def compare_model_sizes(teacher, student, pruned_student):
    # Count parameters
    teacher_params = count_parameters(teacher)
    student_params = count_parameters(student)
    pruned_params = count_parameters(pruned_student)
    
    # Calculate disk size
    teacher_size = calculate_model_size(teacher, "teacher.pth")
    student_size = calculate_model_size(student, "student.pth")
    pruned_size = calculate_model_size(pruned_student, "pruned_student.pth")
    
    # Print comparison
    print("\n--- Model Size Comparison ---")
    print(f"Teacher Model: {teacher_params} parameters, {teacher_size:.2f} MB")
    print(f"Student Model (Before Pruning): {student_params} parameters, {student_size:.2f} MB")
    print(f"Student Model (After Pruning): {pruned_params} parameters, {pruned_size:.2f} MB")
    
    # Calculate compression ratio
    compression_ratio = student_size / pruned_size
    print(f"\nCompression Ratio: {compression_ratio:.2f}x")

In [18]:
def train_model(model, train_loader, val_loader, epochs=10, lr=0.001, patience=3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    
    best_val_accuracy = 0.0
    best_model_state = None
    patience_counter = 0  # Counter for early stopping
    
    for epoch in range(epochs):
        print(epoch)
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        # Evaluate on the validation set
        val_accuracy = evaluate(model, val_loader, device)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f} | Val Accuracy: {val_accuracy:.2f}%")
        
        # Early stopping logic
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = model.state_dict()
            patience_counter = 0  # Reset patience counter
            torch.save(model.state_dict(), 'best_teacher_model.pth')  # Save the best model
            print(f" New best model saved with validation accuracy: {best_val_accuracy:.2f}%")
        else:
            patience_counter += 1
            print(f" No improvement in validation accuracy ({patience_counter}/{patience})")
            
            # Stop training if no improvement for 'patience' epochs
            if patience_counter >= patience:
                print(f"\nEarly stopping triggered! No improvement for {patience} epochs.")
                break
    
    # Load the best model state
    model.load_state_dict(torch.load('best_teacher_model.pth'))
    print("\nLoading the best model for final evaluation.")
    
    # Evaluate on the test set
    test_accuracy = evaluate(model, test_loader, device)
    print(f"Test Accuracy with Best Model: {test_accuracy:.2f}%")
    
    return model



In [19]:
def compute_gradient_importance(
    teacher, student, data_loader, device, temperature=4.0, alpha=0.5, beta_prob=0.5, accumulation_epochs=3
):
    importance_scores = {}

    # Initialize importance score storage for conv layer weights only
    for name, param in student.named_parameters():
        if 'weight' in name and len(param.shape) == 4:  # Conv weights only
            importance_scores[name] = torch.zeros_like(param.data, device=device)

    teacher.to(device).eval()
    student.to(device).train()

    # Add momentum for gradient accumulation smoothing
    momentum = 0.9  # Controls exponential moving average
    accumulated_batches = 0  # Track for bias correction

    for epoch in range(accumulation_epochs):
        print(f"Accumulation Epoch {epoch+1}/{accumulation_epochs}")
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            student.zero_grad()

            with torch.no_grad():
                teacher_logits = teacher(inputs)

            student_logits = student(inputs)

            # Temperature scaling
            student_logits_temp = student_logits / temperature
            teacher_logits_temp = teacher_logits / temperature


            # Compute losses
            distillation_loss = cakld_loss(student_logits_temp, teacher_logits_temp, beta_prob) * (temperature ** 2)
            ce_loss = F.cross_entropy(student_logits, labels)
            loss = alpha * distillation_loss + (1 - alpha) * ce_loss

            # Modified backward propagation
            loss.backward()

            # Accumulate importance scores with parameter-gradient product
            accumulated_batches += 1
            for name, param in student.named_parameters():
                if name in importance_scores and param.grad is not None:
                    # Key modification: Use parameter-gradient product magnitude
                    grad_product = (param.data * param.grad).abs_()
                    
                    # Exponential moving average with bias correction
                    if accumulated_batches == 1:
                        importance_scores[name] = grad_product
                    else:
                        importance_scores[name] = momentum * importance_scores[name] + (1 - momentum) * grad_product

    # Apply bias correction for EMA
    for name in importance_scores:
        importance_scores[name] /= (1 - momentum**accumulated_batches)

    return importance_scores

In [20]:
def gradient_based_global_prune(model, importance_scores, prune_ratio=0.95):
    all_scores = torch.cat([score.flatten() for score in importance_scores.values()])
    threshold = torch.topk(all_scores, k=int(prune_ratio * all_scores.numel()), largest=False)[0][-1]

    for name, param in model.named_parameters():
        if name in importance_scores:
            mask = (importance_scores[name] > threshold).float()
            param.data.mul_(mask)

    return model


In [21]:
import torch
import torch.nn.functional as F
import torch.optim as optim

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

def retrain_with_sparsity(student, train_loader, val_loader, epochs=5, save_path="retrained_student_model.pt", patience=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    optimizer = optim.SGD(student.parameters(), lr=0.01, momentum=0.9)

    # 1. Store masks AND zero momentum buffers for pruned weights
    masks = {}
    for name, param in student.named_parameters():
        if 'weight' in name and param.dim() == 4:  # Consider only conv layers
            mask = (param != 0).float().to(device)
            masks[name] = mask
            # Zero momentum buffers for pruned weights
            if optimizer.state.get(param, None) and 'momentum_buffer' in optimizer.state[param]:
                optimizer.state[param]['momentum_buffer'] *= mask

    student = student.to(device)
    best_val_acc = 0.0
    best_model = None
    patience_counter = 0  # Counter for early stopping

    # 2. Add gradient clipping to prevent NaN
    max_grad_norm = 1.0

    for epoch in range(epochs):
        student.train()
        total_loss = 0.0
        correct, total = 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = student(inputs)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()

            # Apply masks to gradients
            for name, param in student.named_parameters():
                if name in masks:
                    param.grad.data *= masks[name]

            # Gradient clipping before optimizer step
            torch.nn.utils.clip_grad_norm_(student.parameters(), max_grad_norm)

            optimizer.step()

            # Reapply masks and update momentum buffers
            for name, param in student.named_parameters():
                if name in masks:
                    param.data *= masks[name]
                    if optimizer.state.get(param, None) and 'momentum_buffer' in optimizer.state[param]:
                        optimizer.state[param]['momentum_buffer'] *= masks[name]

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / len(train_loader)
        train_acc = 100.0 * correct / total

        # Validation phase
        student.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = student(inputs)
                loss = F.cross_entropy(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_correct += predicted.eq(labels).sum().item()
                val_total += labels.size(0)

        val_loss /= len(val_loader)
        val_acc = 100.0 * val_correct / val_total

        # Track best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = student.state_dict()
            torch.save(best_model, save_path)
            patience_counter = 0  # Reset patience counter
            print(f"New best model saved with Val Accuracy: {best_val_acc:.2f}%")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}. No improvement for {patience} epochs.")
                break  # Stop training

        # Print results
        sparsity = calculate_sparsity(student)
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"Validation Loss: {val_loss:.4f} | Validation Acc: {val_acc:.2f}% | Sparsity: {sparsity*100:.2f}%\n")

    print(f"Best Validation Accuracy: {best_val_acc:.2f}% | Best Model Saved at: {save_path}")
    return student

In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time

# KD training with CA-KLD loss and mask-based momentum handling
def retrain_with_KD(teacher, student, train_loader, val_loader, epochs=50,
                    temperature=5.0, alpha=0.5, beta_prob=0.5, patience=5,
                    save_path="student_before_pruning.pth"):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    optimizer = optim.SGD(student.parameters(), lr=0.01, momentum=0.9)

    # 1. Store masks and zero momentum buffers
    masks = {}
    for name, param in student.named_parameters():
        if 'weight' in name and param.dim() == 4:
            mask = (param != 0).float().to(device)
            masks[name] = mask
            if optimizer.state.get(param, None) and 'momentum_buffer' in optimizer.state[param]:
                optimizer.state[param]['momentum_buffer'] *= mask

    teacher = teacher.to(device).eval()
    student = student.to(device)

    best_val_acc = 0.0
    best_model_state = None
    patience_counter = 0
    start_time = time.time()

    for epoch in range(epochs):
        student.train()
        total_loss, correct, total = 0.0, 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.no_grad():
                teacher_logits = teacher(inputs)

            student_logits = student(inputs)

            # Apply temperature
            teacher_logits_temp = teacher_logits / temperature
            student_logits_temp = student_logits / temperature

            # Logits normalization
            teacher_logits_temp = normalize(teacher_logits_temp)
            student_logits_temp = normalize(student_logits_temp)


            # CA-KLD loss
            kd_loss = cakld_loss(student_logits_temp, teacher_logits_temp, beta_prob) * (temperature ** 2)
            ce_loss = F.cross_entropy(student_logits, labels)

            loss = alpha * kd_loss + (1 - alpha) * ce_loss
            loss.backward()
            optimizer.step()

            # Reapply masks and update momentum
            for name, param in student.named_parameters():
                if name in masks:
                    param.data *= masks[name]
                    if optimizer.state.get(param, None) and 'momentum_buffer' in optimizer.state[param]:
                        optimizer.state[param]['momentum_buffer'] *= masks[name]

            total_loss += loss.item()
            _, predicted = student_logits.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / len(train_loader)
        train_acc = 100.0 * correct / total

        # Validation
        student.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = student(inputs)
                loss = F.cross_entropy(outputs, labels)
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_correct += predicted.eq(labels).sum().item()
                val_total += labels.size(0)

        val_loss /= len(val_loader)
        val_acc = 100.0 * val_correct / val_total
        sparsity = calculate_sparsity(student) * 100.0  # Assuming this function is defined elsewhere

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | Sparsity: {sparsity:.2f}%")

        # Early stopping logic
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = student.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}. No improvement for {patience} epochs.")
                break

    # Restore and save best model
    student.load_state_dict(best_model_state)
    torch.save(student.state_dict(), save_path)
    print(f"Student model saved before pruning at: {save_path}")
    total_time = time.time() - start_time
    print(f"Total Training Time: {total_time // 60:.0f}m {total_time % 60:.0f}s")

    return student

In [23]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Training function with KD + CA-KLD and logits normalization
def train_kd_pruning(teacher, student, train_loader, val_loader, epochs=50, temperature=5.0, alpha=0.5,
                     beta_prob=0.5, patience=5, save_path="student_before_pruning.pth"):
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    optimizer = optim.SGD(student.parameters(), lr=0.01, momentum=0.9)

    teacher = teacher.to(device)
    student = student.to(device)
    teacher.eval()  # Freeze teacher

    best_val_acc = 0.0
    best_model_state = None
    patience_counter = 0
    start_time = time.time()

    for epoch in range(epochs):
        student.train()
        total_loss = 0.0
        correct, total = 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            with torch.no_grad():
                teacher_logits = teacher(inputs)

            student_logits = student(inputs)

            # Temperature scaling
            teacher_logits_temp = teacher_logits / temperature
            student_logits_temp = student_logits / temperature

            # Logits normalization
            teacher_logits_temp = normalize(teacher_logits_temp)
            student_logits_temp = normalize(student_logits_temp)

            # CA-KLD loss (normalized logits)
            distillation_loss = cakld_loss(student_logits_temp, teacher_logits_temp, beta_prob) * (temperature ** 2)

            # Cross-entropy loss
            ground_truth_loss = F.cross_entropy(student_logits, labels)

            # Combined loss
            loss = alpha * distillation_loss + (1 - alpha) * ground_truth_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = student_logits.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / len(train_loader)
        train_acc = 100.0 * correct / total

        # Validation accuracy
        val_acc = evaluate(student, val_loader, device)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | "
              f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = student.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}. No improvement for {patience} epochs.")
                break

    # Load best model state and save
    student.load_state_dict(best_model_state)
    torch.save(student.state_dict(), save_path)
    print(f"Student model saved before pruning at: {save_path}")

    total_time = time.time() - start_time
    print(f"Total Training Time: {total_time // 60:.0f}m {total_time % 60:.0f}s")

    return student

In [24]:
# Load pretrained ResNet-18 (Student Model)
student = models.resnet18(pretrained=True)
# Modify the final fully connected layer for 100 classes (CIFAR-100)
student.fc = nn.Linear(student.fc.in_features, 100)
student = student.to(device)

In [25]:

student = train_kd_pruning(
    teacher, student, train_loader, val_loader,
    epochs=1, temperature=5.0, alpha=0.5,beta_prob=0.5, patience=5,save_path="student_before_pruning.pth"
)


Epoch 1/1 | Train Loss: 9.8385 | Train Acc: 53.51% | Val Acc: 61.96%
Student model saved before pruning at: student_before_pruning.pth
Total Training Time: 2m 17s


In [26]:
# Calculate sparsity
sparsity = calculate_sparsity(student)
print(f"Sparsity Before Pruning: {sparsity * 100:.2f}%")

teacher_accuracy = evaluate(teacher, test_loader, device)
student_accuracy = evaluate(student, test_loader, device)
print(f"Teacher Model Test Accuracy: {teacher_accuracy:.2f}%")
print(f"Student Model Test Accuracy Before Pruning: {student_accuracy:.2f}%")

Sparsity Before Pruning: 0.00%
Teacher Model Test Accuracy: 80.91%
Student Model Test Accuracy Before Pruning: 62.11%


## 97.7% Sparsity

In [135]:

model_path = '/kaggle/working/student_before_pruning.pth'
# Load the model weights
student.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
# Calculate sparsity


<All keys matched successfully>

In [136]:
sparsity = calculate_sparsity(student)
print(f"Sparsity Before Pruning: {sparsity * 100:.2f}%")

Sparsity Before Pruning: 0.00%


In [137]:
model=student

In [138]:
import torch
import torch.nn.utils.prune as prune
import torch.nn as nn
import copy
torch.manual_seed(42)
original_init = copy.deepcopy(model.state_dict())  #

parameters_to_prune = []
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
        parameters_to_prune.append((module, 'weight'))

# 4️ Apply one-shot global magnitude pruning
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,  # L1 magnitude pruning
    amount=0.5065
)

# Optional: Remove pruning reparameterization so `.weight` becomes pruned tensor
for module, param in parameters_to_prune:
    prune.remove(module, 'weight')

# 5️ Get pruning mask
mask_dict = {k: (v != 0).float() for k, v in model.state_dict().items()}

# 6️ Reset remaining weights to their original initialization values
reset_state = {}
for k in original_init.keys():
    if k in mask_dict:
        reset_state[k] = original_init[k] * mask_dict[k]  # keep init where mask=1
    else:
        reset_state[k] = original_init[k]  # non-pruned params like biases


In [139]:
model.load_state_dict(reset_state)

<All keys matched successfully>

In [140]:
sparsity = calculate_sparsity(model)
print(f"Sparsity after Pruning: {sparsity * 100:.2f}%")

Sparsity after Pruning: 50.63%


In [141]:

start_time = time.time()
pruned_student = retrain_with_KD(
    teacher, model, train_loader, val_loader,
    epochs=50, temperature=3.0, alpha=0.7, beta_prob=0.5,patience=5,save_path="pruned_student_retrain_KD_90%.pth"
)
end_time = time.time()
elapsed_time = end_time - start_time

print(f"Retraining completed in {elapsed_time / 60:.2f} minutes ({elapsed_time:.2f} seconds)")

Epoch 1/50 | Train Loss: 1.6330 | Train Acc: 79.53% | Val Loss: 0.8516 | Val Acc: 76.21% | Sparsity: 50.53%
Epoch 2/50 | Train Loss: 1.0432 | Train Acc: 86.22% | Val Loss: 0.7652 | Val Acc: 77.95% | Sparsity: 50.53%
Epoch 3/50 | Train Loss: 0.7522 | Train Acc: 90.12% | Val Loss: 0.7206 | Val Acc: 79.39% | Sparsity: 50.53%
Epoch 4/50 | Train Loss: 0.5719 | Train Acc: 92.75% | Val Loss: 0.6916 | Val Acc: 79.72% | Sparsity: 50.53%
Epoch 5/50 | Train Loss: 0.4642 | Train Acc: 94.37% | Val Loss: 0.6792 | Val Acc: 80.30% | Sparsity: 50.53%
Epoch 6/50 | Train Loss: 0.4000 | Train Acc: 95.25% | Val Loss: 0.6766 | Val Acc: 80.54% | Sparsity: 50.53%
Epoch 7/50 | Train Loss: 0.3581 | Train Acc: 95.85% | Val Loss: 0.6630 | Val Acc: 80.95% | Sparsity: 50.53%
Epoch 8/50 | Train Loss: 0.3256 | Train Acc: 96.21% | Val Loss: 0.6643 | Val Acc: 81.03% | Sparsity: 50.53%
Epoch 9/50 | Train Loss: 0.3018 | Train Acc: 96.37% | Val Loss: 0.6606 | Val Acc: 81.16% | Sparsity: 50.53%
Epoch 10/50 | Train Loss: 0.

In [142]:
student_accuracy = evaluate(pruned_student, test_loader, device)
print(f"Pruned Student Model Test Accuracy(After Retrain): {student_accuracy:.2f}%")

Pruned Student Model Test Accuracy(After Retrain): 80.36%


In [32]:

model_path = '/kaggle/working/student_before_pruning.pth'
# Load the model weights
student.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
# Calculate sparsity


model=student
import torch
import torch.nn.utils.prune as prune
import torch.nn as nn
import copy
torch.manual_seed(42)
original_init = copy.deepcopy(model.state_dict())  #

parameters_to_prune = []
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
        parameters_to_prune.append((module, 'weight'))

# 4️ Apply one-shot global magnitude pruning
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,  # L1 magnitude pruning
    amount=0.9843
)

# Optional: Remove pruning reparameterization so `.weight` becomes pruned tensor
for module, param in parameters_to_prune:
    prune.remove(module, 'weight')

# 5️ Get pruning mask
mask_dict = {k: (v != 0).float() for k, v in model.state_dict().items()}

# 6️ Reset remaining weights to their original initialization values
reset_state = {}
for k in original_init.keys():
    if k in mask_dict:
        reset_state[k] = original_init[k] * mask_dict[k]  # keep init where mask=1
    else:
        reset_state[k] = original_init[k]  # non-pruned params like biases


sparsity = calculate_sparsity(model)
print(f"Sparsity Before Pruning: {sparsity * 100:.2f}%")
model.load_state_dict(reset_state)

Sparsity Before Pruning: 98.39%


<All keys matched successfully>

In [33]:

start_time = time.time()
pruned_student = retrain_with_KD(
    teacher, model, train_loader, val_loader,
    epochs=50, temperature=3.0, alpha=980.7, beta_prob=0.5,patience=1,save_path="pruned_student_retrain_KD_90%.pth"
)
end_time = time.time()
elapsed_time = end_time - start_time

print(f"Retraining completed in {elapsed_time / 60:.2f} minutes ({elapsed_time:.2f} seconds)")

Epoch 1/50 | Train Loss: 5.9238 | Train Acc: 47.06% | Val Loss: 1.5592 | Val Acc: 57.79% | Sparsity: 98.01%
Epoch 2/50 | Train Loss: 3.8169 | Train Acc: 62.35% | Val Loss: 1.4093 | Val Acc: 61.49% | Sparsity: 98.01%
Epoch 3/50 | Train Loss: 3.2190 | Train Acc: 67.41% | Val Loss: 1.2975 | Val Acc: 64.91% | Sparsity: 98.01%
Epoch 4/50 | Train Loss: 2.8697 | Train Acc: 70.39% | Val Loss: 1.2568 | Val Acc: 65.67% | Sparsity: 98.01%
Epoch 5/50 | Train Loss: 2.6262 | Train Acc: 72.55% | Val Loss: 1.2435 | Val Acc: 65.54% | Sparsity: 98.01%
Early stopping triggered at epoch 5. No improvement for 1 epochs.
Student model saved before pruning at: pruned_student_retrain_KD_90%.pth
Total Training Time: 11m 22s
Retraining completed in 11.36 minutes (681.71 seconds)


In [34]:
student_accuracy = evaluate(pruned_student, test_loader, device)
print(f"Pruned Student Model Test Accuracy(After Retrain): {student_accuracy:.2f}%")

Pruned Student Model Test Accuracy(After Retrain): 65.20%


In [43]:

model_path = '/kaggle/working/student_before_pruning.pth'
# Load the model weights
student.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
# Calculate sparsity


model=student
import torch
import torch.nn.utils.prune as prune
import torch.nn as nn
import copy
torch.manual_seed(42)
original_init = copy.deepcopy(model.state_dict())  #

parameters_to_prune = []
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
        parameters_to_prune.append((module, 'weight'))

# 4️ Apply one-shot global magnitude pruning
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,  # L1 magnitude pruning
    amount=0.7488
)

# Optional: Remove pruning reparameterization so `.weight` becomes pruned tensor
for module, param in parameters_to_prune:
    prune.remove(module, 'weight')

# 5️ Get pruning mask
mask_dict = {k: (v != 0).float() for k, v in model.state_dict().items()}

# 6️ Reset remaining weights to their original initialization values
reset_state = {}
for k in original_init.keys():
    if k in mask_dict:
        reset_state[k] = original_init[k] * mask_dict[k]  # keep init where mask=1
    else:
        reset_state[k] = original_init[k]  # non-pruned params like biases


sparsity = calculate_sparsity(model)
print(f"Sparsity Before Pruning: {sparsity * 100:.2f}%")
model.load_state_dict(reset_state)

Sparsity Before Pruning: 74.85%


<All keys matched successfully>

In [44]:

start_time = time.time()
pruned_student = retrain_with_KD(
    teacher, model, train_loader, val_loader,
    epochs=50, temperature=3.0, alpha=0.7, beta_prob=0.5,patience=5,save_path="pruned_student_retrain_KD_90%.pth"
)
end_time = time.time()
elapsed_time = end_time - start_time

print(f"Retraining completed in {elapsed_time / 60:.2f} minutes ({elapsed_time:.2f} seconds)")

Epoch 1/50 | Train Loss: 1.7809 | Train Acc: 77.98% | Val Loss: 0.8646 | Val Acc: 75.26% | Sparsity: 74.69%
Epoch 2/50 | Train Loss: 1.1862 | Train Acc: 84.46% | Val Loss: 0.7680 | Val Acc: 78.34% | Sparsity: 74.69%
Epoch 3/50 | Train Loss: 0.8809 | Train Acc: 88.47% | Val Loss: 0.7481 | Val Acc: 78.74% | Sparsity: 74.69%
Epoch 4/50 | Train Loss: 0.6906 | Train Acc: 91.18% | Val Loss: 0.7178 | Val Acc: 80.05% | Sparsity: 74.69%
Epoch 5/50 | Train Loss: 0.5671 | Train Acc: 93.09% | Val Loss: 0.7023 | Val Acc: 80.23% | Sparsity: 74.69%
Epoch 6/50 | Train Loss: 0.4815 | Train Acc: 94.25% | Val Loss: 0.7136 | Val Acc: 79.91% | Sparsity: 74.69%
Epoch 7/50 | Train Loss: 0.4255 | Train Acc: 94.96% | Val Loss: 0.6932 | Val Acc: 80.59% | Sparsity: 74.69%
Epoch 8/50 | Train Loss: 0.3839 | Train Acc: 95.69% | Val Loss: 0.7047 | Val Acc: 80.37% | Sparsity: 74.69%
Epoch 9/50 | Train Loss: 0.3559 | Train Acc: 95.94% | Val Loss: 0.6914 | Val Acc: 80.80% | Sparsity: 74.69%
Epoch 10/50 | Train Loss: 0.

In [45]:
student_accuracy = evaluate(pruned_student, test_loader, device)
print(f"Pruned Student Model Test Accuracy(After Retrain): {student_accuracy:.2f}%")

Pruned Student Model Test Accuracy(After Retrain): 79.63%


In [29]:

model_path = '/kaggle/working/student_before_pruning.pth'
# Load the model weights
student.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
# Calculate sparsity


model=student
import torch
import torch.nn.utils.prune as prune
import torch.nn as nn
import copy
torch.manual_seed(42)
original_init = copy.deepcopy(model.state_dict())  #

parameters_to_prune = []
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
        parameters_to_prune.append((module, 'weight'))

# 4️ Apply one-shot global magnitude pruning
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,  # L1 magnitude pruning
    amount=0.8082
)

# Optional: Remove pruning reparameterization so `.weight` becomes pruned tensor
for module, param in parameters_to_prune:
    prune.remove(module, 'weight')

# 5️ Get pruning mask
mask_dict = {k: (v != 0).float() for k, v in model.state_dict().items()}

# 6️ Reset remaining weights to their original initialization values
reset_state = {}
for k in original_init.keys():
    if k in mask_dict:
        reset_state[k] = original_init[k] * mask_dict[k]  # keep init where mask=1
    else:
        reset_state[k] = original_init[k]  # non-pruned params like biases


sparsity = calculate_sparsity(model)
print(f"Sparsity Before Pruning: {sparsity * 100:.2f}%")
model.load_state_dict(reset_state)

Sparsity Before Pruning: 80.79%


<All keys matched successfully>

In [30]:

start_time = time.time()
pruned_student = retrain_with_KD(
    teacher, model, train_loader, val_loader,
    epochs=50, temperature=3.0, alpha=0.7, beta_prob=0.5,patience=5,save_path="pruned_student_retrain_KD_90%.pth"
)
end_time = time.time()
elapsed_time = end_time - start_time

print(f"Retraining completed in {elapsed_time / 60:.2f} minutes ({elapsed_time:.2f} seconds)")

Epoch 1/50 | Train Loss: 1.9435 | Train Acc: 76.81% | Val Loss: 0.9008 | Val Acc: 74.47% | Sparsity: 80.60%
Epoch 2/50 | Train Loss: 1.2846 | Train Acc: 83.39% | Val Loss: 0.7886 | Val Acc: 77.42% | Sparsity: 80.60%
Epoch 3/50 | Train Loss: 0.9640 | Train Acc: 87.53% | Val Loss: 0.7741 | Val Acc: 77.30% | Sparsity: 80.60%
Epoch 4/50 | Train Loss: 0.7508 | Train Acc: 90.54% | Val Loss: 0.7418 | Val Acc: 78.75% | Sparsity: 80.60%
Epoch 5/50 | Train Loss: 0.6137 | Train Acc: 92.39% | Val Loss: 0.7345 | Val Acc: 78.52% | Sparsity: 80.60%
Epoch 6/50 | Train Loss: 0.5254 | Train Acc: 93.69% | Val Loss: 0.7301 | Val Acc: 79.03% | Sparsity: 80.60%
Epoch 7/50 | Train Loss: 0.4621 | Train Acc: 94.58% | Val Loss: 0.7325 | Val Acc: 78.96% | Sparsity: 80.60%
Epoch 8/50 | Train Loss: 0.4192 | Train Acc: 95.06% | Val Loss: 0.7309 | Val Acc: 79.06% | Sparsity: 80.60%
Epoch 9/50 | Train Loss: 0.3820 | Train Acc: 95.56% | Val Loss: 0.7152 | Val Acc: 79.62% | Sparsity: 80.60%
Epoch 10/50 | Train Loss: 0.

In [31]:
student_accuracy = evaluate(pruned_student, test_loader, device)
print(f"Pruned Student Model Test Accuracy(After Retrain): {student_accuracy:.2f}%")

Pruned Student Model Test Accuracy(After Retrain): 79.11%


In [32]:

model_path = '/kaggle/working/student_before_pruning.pth'
# Load the model weights
student.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
# Calculate sparsity


model=student
import torch
import torch.nn.utils.prune as prune
import torch.nn as nn
import copy
torch.manual_seed(42)
original_init = copy.deepcopy(model.state_dict())  #

parameters_to_prune = []
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
        parameters_to_prune.append((module, 'weight'))

# 4️ Apply one-shot global magnitude pruning
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,  # L1 magnitude pruning
    amount=0.5075
)

# Optional: Remove pruning reparameterization so `.weight` becomes pruned tensor
for module, param in parameters_to_prune:
    prune.remove(module, 'weight')

# 5️ Get pruning mask
mask_dict = {k: (v != 0).float() for k, v in model.state_dict().items()}

# 6️ Reset remaining weights to their original initialization values
reset_state = {}
for k in original_init.keys():
    if k in mask_dict:
        reset_state[k] = original_init[k] * mask_dict[k]  # keep init where mask=1
    else:
        reset_state[k] = original_init[k]  # non-pruned params like biases


sparsity = calculate_sparsity(model)
print(f"Sparsity Before Pruning: {sparsity * 100:.2f}%")
model.load_state_dict(reset_state)

Sparsity Before Pruning: 50.73%


<All keys matched successfully>

In [33]:

start_time = time.time()
pruned_student = retrain_with_KD(
    teacher, model, train_loader, val_loader,
    epochs=50, temperature=3.0, alpha=0.7, beta_prob=0.5,patience=5,save_path="pruned_student_retrain_KD_90%.pth"
)
end_time = time.time()
elapsed_time = end_time - start_time

print(f"Retraining completed in {elapsed_time / 60:.2f} minutes ({elapsed_time:.2f} seconds)")

Epoch 1/50 | Train Loss: 1.6810 | Train Acc: 78.94% | Val Loss: 0.8577 | Val Acc: 75.94% | Sparsity: 50.63%
Epoch 2/50 | Train Loss: 1.0601 | Train Acc: 86.06% | Val Loss: 0.7462 | Val Acc: 78.64% | Sparsity: 50.63%
Epoch 3/50 | Train Loss: 0.7503 | Train Acc: 90.34% | Val Loss: 0.7382 | Val Acc: 78.50% | Sparsity: 50.63%
Epoch 4/50 | Train Loss: 0.5694 | Train Acc: 92.95% | Val Loss: 0.7115 | Val Acc: 79.58% | Sparsity: 50.63%
Epoch 5/50 | Train Loss: 0.4655 | Train Acc: 94.30% | Val Loss: 0.6993 | Val Acc: 80.08% | Sparsity: 50.63%
Epoch 6/50 | Train Loss: 0.4005 | Train Acc: 95.25% | Val Loss: 0.6914 | Val Acc: 80.08% | Sparsity: 50.63%
Epoch 7/50 | Train Loss: 0.3587 | Train Acc: 95.74% | Val Loss: 0.6895 | Val Acc: 80.48% | Sparsity: 50.63%
Epoch 8/50 | Train Loss: 0.3273 | Train Acc: 96.15% | Val Loss: 0.6876 | Val Acc: 80.52% | Sparsity: 50.63%
Epoch 9/50 | Train Loss: 0.3033 | Train Acc: 96.32% | Val Loss: 0.6818 | Val Acc: 80.68% | Sparsity: 50.63%
Epoch 10/50 | Train Loss: 0.

In [34]:
student_accuracy = evaluate(pruned_student, test_loader, device)
print(f"Pruned Student Model Test Accuracy(After Retrain): {student_accuracy:.2f}%")

Pruned Student Model Test Accuracy(After Retrain): 80.27%
